<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ZeroToHeroColabCollection/blob/main/gpt_from_scratch_again_recall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Standard Imports
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Hyper Parameters
BATCH_SIZE = 32
CONTEXT_LENGTH = 8
N_EMBD = 10
N_HEADS = 2
N_BLOCKS = 6

In [ ]:
# Opening the Dataset
with open('goft.txt', 'r') as file:
  data = re.sub("[^a-zA-Z0-9,!&.\n ]", "", file.read())

In [ ]:
# Character Mapping
chars = sorted(list(set(data)))
stoi = {char: i for i, char in enumerate(chars)}
itos = dict(zip(stoi.values(), stoi.keys()))
len(chars)

67

In [ ]:
# Creating Encoding and Decoding Functions
encode = lambda text: list(map(lambda char: stoi[char], list(text)))
decode = lambda toks: ''.join(list(map(lambda tok: itos[tok], toks)))
encode('hi!'), decode([48, 49, 2])

([48, 49, 2], 'hi!')

In [ ]:
# Tokenizing Dataset
tokens = torch.tensor(encode(data), dtype = torch.long)
tokens.shape

torch.Size([1593624])

In [ ]:
# Creating Training Testing Data Split
n = int(0.8 * tokens.shape[0])
train_data, val_data = tokens[:n], tokens[n:]

In [ ]:
# Build A Helper Batch Function
def get_batch(data_type):
  cdata = train_data if data_type == 'train' else val_data
  ix = torch.randint(low=0, high = (cdata.shape[0] - CONTEXT_LENGTH - 1), size = (BATCH_SIZE,), dtype = torch.long)
  x, y = torch.stack(tuple(cdata[i: i + CONTEXT_LENGTH] for i in ix)), torch.stack(tuple(cdata[i + 1: i + CONTEXT_LENGTH + 1] for i in ix))
  return x, y

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, n_embd, context_length, n_heads):
    super().__init__()
    self.n_heads = n_heads
    self.keys = nn.Linear(n_embd, n_embd)
    self.queries = nn.Linear(n_embd, n_embd)
    self.values = nn.Linear(n_embd, n_embd)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

  def forward(self, x):
    # Saving x's shape
    B, T, C = x.shape # Target Shape = B, T, C

    # Creating Keys Queries & Values
    k, q, v = self.keys(x), self.queries(x), self.values(x) # B, T, C
    k, q, v = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2), q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2), v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
    # v.shape = B, n_heads, T, C // n_heads

    # Creating Weighted Attention
    wei = q @ k.transpose(-1, -2) # B, n_heads, T, T
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim = -1) # still not sure y last dim

    # Outputting The Attention
    out = torch.transpose(wei @ v, 1, 2)
    out = out.reshape(B, T, C)
    return out

In [ ]:
class FeedForward(nn.Module):
  def __init__ (self, n_embd):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(n_embd, n_embd),
        nn.ReLU()
    )
    self.ln1 = nn.LayerNorm(n_embd)

  def forward(self, x):
    out = self.network(x) + x
    out = self.ln1(out)
    return out + x

In [ ]:
class Block(nn.Module):
  def __init__(self, n_embd, context_length, n_heads):
    super().__init__()
    self.mha = MultiHeadAttention(n_embd, context_length, n_heads)
    self.ln2 = nn.LayerNorm(n_embd)
    self.ffd = FeedForward(n_embd)

  def forward(self, x):
    out = self.mha(x)
    out = self.ln2(out) + x
    out = self.ffd(out)
    return out

In [ ]:
# Creating A Base Model and Building Up From That
class GPT(nn.Module):
  def __init__ (self, vocab_size, n_embd, context_length, n_heads, n_blocks):
    super().__init__()
    self.tok_embd = nn.Embedding(vocab_size, n_embd)
    self.pos_embd = nn.Embedding(context_length, n_embd)
    self.blocks = nn.Sequential(
        *[Block(n_embd, context_length, n_heads) for _ in range(n_blocks)]
    )
    self.lm = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets = None):
    tok_emb = self.tok_embd(x) # B, T, C
    pos_emb = self.pos_embd(torch.arange(x.shape[1])) # T, C
    emb = tok_emb + pos_emb # B, T, C
    prelogits = self.blocks(emb) # B, T, C
    logits = self.lm(prelogits) # 32, 8, 67
    if targets is None:
      return logits
    else:
      loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
      return logits, loss

In [ ]:
# Model
x, y = get_batch('train')
gpt = GPT(len(chars), N_EMBD, CONTEXT_LENGTH, N_HEADS, N_BLOCKS)
logits, loss = gpt(x, y)
logits.shape, loss

(torch.Size([32, 8, 67]), tensor(11.7180, grad_fn=<NllLossBackward0>))